# 10-K Financial Trend Analysis for Microsoft, Tesla, and Apple (2023-2025)

This notebook analyzes key financial statement metrics manually compiled from SEC EDGAR 10-K filings for Microsoft, Tesla, and Apple. The goal is to identify trends and insights that could inform the design of an AI-powered financial chatbot.

## Methodology

The source dataset was compiled from each company's annual 10-K filing in SEC EDGAR for fiscal years 2023, 2024, and 2025. The extracted metrics are total revenue, net income, total assets, total liabilities, and cash flow from operating activities.

Source concepts used from the SEC filing data:

- Total Revenue: `RevenueFromContractWithCustomerExcludingAssessedTax` (Tesla also reports the same annual value under `Revenues`)
- Net Income: `NetIncomeLoss`
- Total Assets: `Assets`
- Total Liabilities: `Liabilities`
- Cash Flow from Operating Activities: `NetCashProvidedByUsedInOperatingActivities`

The raw CSV keeps values in dollars. This notebook converts values to billions for readability while preserving the original source file.

In [ ]:
import os
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".matplotlib"))

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", "{:.2f}".format)
pd.set_option("display.notebook_repr_html", False)
DATA_PATH = Path("financial_10k_data.csv")

metric_cols = [
    "Total Revenue",
    "Net Income",
    "Total Assets",
    "Total Liabilities",
    "Cash Flow from Operating Activities",
]

analysis_view_cols = [
    "Company",
    "Fiscal Year",
    "Total Revenue (USD billions)",
    "Total Revenue Growth (%)",
    "Net Income (USD billions)",
    "Net Income Growth (%)",
    "Total Assets (USD billions)",
    "Total Liabilities (USD billions)",
    "Cash Flow from Operating Activities (USD billions)",
    "Net Margin (%)",
    "Operating Cash Flow Margin (%)",
    "Liabilities to Assets (%)",
]


def prepare_analysis():
    base_df = pd.read_csv(DATA_PATH).sort_values(["Company", "Fiscal Year"]).reset_index(drop=True)
    prepared = base_df.copy()

    for col in metric_cols:
        prepared[f"{col} (USD billions)"] = prepared[col] / 1_000_000_000
        prepared[f"{col} Growth (%)"] = prepared.groupby("Company")[col].pct_change() * 100

    prepared["Net Margin (%)"] = prepared["Net Income"] / prepared["Total Revenue"] * 100
    prepared["Operating Cash Flow Margin (%)"] = prepared["Cash Flow from Operating Activities"] / prepared["Total Revenue"] * 100
    prepared["Liabilities to Assets (%)"] = prepared["Total Liabilities"] / prepared["Total Assets"] * 100
    return base_df, prepared

In [ ]:
df = pd.read_csv(DATA_PATH)
df = df.sort_values(["Company", "Fiscal Year"]).reset_index(drop=True)
df

## Prepare Analysis Columns

To make comparisons easier, the financial metrics are converted to USD billions. Year-over-year growth is calculated within each company so Microsoft, Tesla, and Apple are compared only against their own prior year.

In [ ]:
df, analysis = prepare_analysis()
analysis[analysis_view_cols].round(2)

## Revenue and Profitability Trends

In [ ]:
if "analysis" not in globals():
    df, analysis = prepare_analysis()

pivot_revenue = analysis.pivot(index="Fiscal Year", columns="Company", values="Total Revenue (USD billions)")
pivot_net_income = analysis.pivot(index="Fiscal Year", columns="Company", values="Net Income (USD billions)")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
pivot_revenue.plot(ax=axes[0], marker="o")
axes[0].set_title("Total Revenue by Company")
axes[0].set_xlabel("Fiscal Year")
axes[0].set_ylabel("USD billions")
axes[0].grid(True, alpha=0.3)

pivot_net_income.plot(ax=axes[1], marker="o")
axes[1].set_title("Net Income by Company")
axes[1].set_xlabel("Fiscal Year")
axes[1].set_ylabel("USD billions")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()

In [ ]:
if "analysis" not in globals():
    df, analysis = prepare_analysis()

growth_cols = [
    "Company",
    "Fiscal Year",
    "Total Revenue Growth (%)",
    "Net Income Growth (%)",
    "Cash Flow from Operating Activities Growth (%)",
]
analysis[growth_cols].round(2)

## Balance Sheet and Cash Flow Indicators

In [ ]:
if "analysis" not in globals():
    df, analysis = prepare_analysis()

latest = analysis[analysis["Fiscal Year"] == 2025].copy()
latest_summary = latest[[
    "Company",
    "Total Revenue (USD billions)",
    "Net Income (USD billions)",
    "Cash Flow from Operating Activities (USD billions)",
    "Net Margin (%)",
    "Operating Cash Flow Margin (%)",
    "Liabilities to Assets (%)",
]].sort_values("Total Revenue (USD billions)", ascending=False)
latest_summary.round(2)

In [ ]:
if "analysis" not in globals():
    df, analysis = prepare_analysis()

balance = analysis.pivot(index="Fiscal Year", columns="Company", values="Liabilities to Assets (%)")
balance.plot(figsize=(8, 4), marker="o")
plt.title("Liabilities as a Percentage of Assets")
plt.xlabel("Fiscal Year")
plt.ylabel("Liabilities / Assets (%)")
plt.grid(True, alpha=0.3)
plt.tight_layout()

## Compound Growth from 2023 to 2025

The period covers two year-over-year intervals: 2023 to 2024 and 2024 to 2025. The CAGR calculation helps summarize the full-period trajectory for each metric.

In [ ]:
if "analysis" not in globals():
    df, analysis = prepare_analysis()


def cagr(first, last, periods):
    return ((last / first) ** (1 / periods) - 1) * 100

cagr_rows = []
for company, group in analysis.groupby("Company"):
    start = group[group["Fiscal Year"] == 2023].iloc[0]
    end = group[group["Fiscal Year"] == 2025].iloc[0]
    row = {"Company": company}
    for col in metric_cols:
        row[f"{col} CAGR 2023-2025 (%)"] = cagr(start[col], end[col], 2)
    cagr_rows.append(row)

cagr_df = pd.DataFrame(cagr_rows)
cagr_df.round(2)

## Key Findings

- **Microsoft showed the strongest broad-based expansion.** Revenue rose from $211.9B in 2023 to $281.7B in 2025, while net income increased from $72.4B to $101.8B. Operating cash flow also grew strongly, reaching $136.2B in 2025. This indicates a high-quality growth profile: revenue, profit, assets, and operating cash generation all moved upward together.
- **Tesla showed pressure in profitability despite a larger asset base.** Revenue was roughly flat in 2024 and declined in 2025, while net income fell from $15.0B in 2023 to $3.8B in 2025. Assets continued to grow, suggesting expansion or investment continued even as profitability compressed.
- **Apple remained highly profitable and rebounded in 2025.** Revenue increased from $383.3B in 2023 to $416.2B in 2025. Net income dipped in 2024 but rose sharply to $112.0B in 2025. Apple also reduced total liabilities in 2025, lowering liabilities as a percentage of assets.
- **Cash flow quality differs across companies.** Microsoft's operating cash flow grew each year. Apple's operating cash flow stayed very large but declined in 2025 after a 2024 increase. Tesla's operating cash flow remained positive and relatively stable, but the gap between cash flow and net income highlights the need for deeper chatbot explanations around non-cash items, working capital, and capital intensity.

## Implications for an AI-Powered Financial Chatbot

A useful financial chatbot should do more than repeat figures from filings. These trends suggest several product requirements:

- The chatbot should explain **why growth metrics differ** across revenue, net income, assets, liabilities, and operating cash flow.
- It should provide **company-specific context**, such as different fiscal year ends and industry economics.
- It should flag **trend divergence**, for example Tesla's growing assets alongside falling net income, or Apple's rising net income alongside lower operating cash flow in 2025.
- It should support **source-grounded answers** by linking responses back to SEC filing dates, accession numbers, and filing URLs.
- It should present financial values in human-readable units while retaining raw-dollar precision for calculations.

Overall, Microsoft appears to have the most consistent growth profile over this period, Apple remains the largest revenue generator with strong profitability, and Tesla shows the greatest volatility and need for explanatory context.

## Source Filing References

In [ ]:
if "df" not in globals():
    df, analysis = prepare_analysis()

df[["Company", "Fiscal Year", "10-K Filing Date", "SEC Accession Number", "SEC Filing URL"]]